# Send로 동적 팬아웃

검색된 문서 수가 실행할 때마다 달라도 각 문서를 같은 노드로 보내고 결과를 모을 수 있을까요?

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** dispatch가 반환하는 Send('inspect_document', {'document': doc}) 하나는 목적지 이름과 그 실행 전용 입력을 묶습니다. 문서가 세 건이면 Send도 세 개가 생깁니다. 각 실행은 results에 한 항목을 반환하고, 상위 상태의 리듀서가 여러 반환값을 하나의 목록으로 합칩니다.

**다른 상황에 적용.** 상품 리뷰를 언어별로 분류하거나 여러 PDF의 각 페이지를 같은 검사 함수에 보내는 흐름에도 적용할 수 있습니다. 항목 수가 실행 때 정해지고 각 항목을 같은 방식으로 처리한다는 조건이 같기 때문입니다.

**비교와 경계.** Send는 동적인 작업 수를 표현하지만 완료 순서나 성능 향상을 보장하지 않습니다. 출력 순서가 중요하면 ID와 원래 위치를 함께 보내고 마지막에 정렬해야 합니다. 모든 항목이 앞 항목의 결과를 필요로 한다면 팬아웃보다 순차 엣지가 알맞습니다.


## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 실행합니다. 설치 셀은 이 노트북에 필요한 버전을 고정합니다.

## 패키지 설치와 버전 확인

필요한 패키지를 고정 버전으로 설치합니다. 출력에서 LangGraph 버전 또는 API 키의 존재 여부만 확인합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [1]:
# 노트북을 독립 실행할 수 있도록 패키지와 환경을 준비합니다.
%pip install -q langgraph==1.2.11 langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 scikit-learn==1.9.0
from importlib.metadata import version
print('Python 패키지 준비 완료')
print('langgraph', version('langgraph'))

Note: you may need to restart the kernel to use updated packages.
Python 패키지 준비 완료
langgraph 1.2.11


## 전체 상태와 작업 상태 정의

다음 코드에서 전체 상태와 작업 상태 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

**이 셀의 코드 읽기.** 이 셀을 실행하면 `OverallState`, `DocumentState` 클래스 정의가 준비됩니다. 클래스 본문은 정의 시 처리되지만 인스턴스 값은 나중에 객체를 만들 때 생깁니다. `Annotated[value_type, reducer]`는 값의 형식과 새 갱신을 합칠 함수를 함께 표시합니다.


In [2]:
# 전체 상태와 작업 상태 정의에 사용할 상태 형식과 함수를 선언합니다.
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class OverallState(TypedDict):
    documents: list[dict]
    results: Annotated[list[dict], operator.add]

class DocumentState(TypedDict):
    document: dict

## Send 목록과 작업 노드 정의

다음 코드에서 Send 목록과 작업 노드 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

**이 셀의 코드 읽기.** `dispatch(state: OverallState)`, `inspect_document(state: DocumentState)`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `StateGraph(StateType)`은 상태 형식을 입력으로 받아 그래프 설계 객체를 만듭니다. `add_node(name, function)`은 그래프 이름과 실제 실행할 함수를 연결하며, 이 호출만으로 함수 본문이 실행되지는 않습니다. `add_edge(from, to)`는 두 노드의 고정 이동 순서를 등록합니다.


In [3]:
# Send 목록과 작업 노드 정의에 사용할 상태 형식과 함수를 선언합니다.
def dispatch(state: OverallState) -> list[Send]:
    return [Send('inspect_document', {'document': doc}) for doc in state['documents']]

def inspect_document(state: DocumentState) -> dict:
    doc = state['document']
    return {'results': [{'id': doc['id'], 'length': len(doc['text'])}]}

builder = StateGraph(OverallState)
builder.add_node('inspect_document', inspect_document)
builder.add_conditional_edges(START, dispatch)
builder.add_edge('inspect_document', END)
graph = builder.compile()

## 세 문서로 팬아웃

세 문서로 팬아웃 단계를 실행합니다. 출력된 상태 값과 코드에서 반환한 키를 서로 대조합니다.

**이 셀의 코드 읽기.** `invoke(input)`가 나타나는 줄에서 실제 그래프 또는 모델 실행이 시작되고 반환값이 왼쪽 변수에 저장됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [4]:
# 세 문서로 팬아웃 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
documents = [
    {'id': 'E01', 'text': '대출은 7권까지 가능합니다.'},
    {'id': 'E02', 'text': '대출 기간은 14일입니다.'},
    {'id': 'E03', 'text': '반납함은 출입구에 있습니다.'},
]
result = graph.invoke({'documents': documents, 'results': []})
ordered = sorted(result['results'], key=lambda item: item['id'])
print(ordered)
assert len(ordered) == 3

[{'id': 'E01', 'length': 15}, {'id': 'E02', 'length': 14}, {'id': 'E03', 'length': 15}]


## 입력 수 하나로 비교

입력 수 하나로 비교 조건으로 값을 하나만 변경합니다. 나란히 출력된 두 결과에서 달라진 항목을 확인합니다.

**이 셀의 코드 읽기.** `invoke(input)`가 나타나는 줄에서 실제 그래프 또는 모델 실행이 시작되고 반환값이 왼쪽 변수에 저장됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [5]:
# 한 조건만 바꾸고 나머지 입력을 유지해 결과 차이를 비교합니다.
single = graph.invoke({'documents': documents[:1], 'results': []})
print(single['results'])
assert len(single['results']) == 1

[{'id': 'E01', 'length': 15}]
